In [ ]:
!pip install geopandas shapely requests tqdm -q
!pip install boto3 -q
!pip install webdataset

import geopandas as gpd
import pandas as pd
import numpy as np
import requests
import time
import random
from pathlib import Path
from shapely.geometry import Point
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import os

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

In [ ]:
import boto3
import webdataset as wds
import json
import io
import os
from tqdm.notebook import tqdm
from google.colab import userdata

# Configuration
BUCKET = 'geolocation-transformer-data'
REGION = 'us-west-2'

# Initialize AWS session from Colab Secrets
session = boto3.Session(
    aws_access_key_id=userdata.get('AWS_ACCESS_KEY_ID'),
    aws_secret_access_key=userdata.get('AWS_SECRET_ACCESS_KEY'),
    region_name=REGION,
)
s3_client = session.client('s3')
dynamodb = session.resource('dynamodb')
table = dynamodb.Table('GeolocationMetadata')

def upload_and_delete(fname):
    shard_name = os.path.basename(fname)
    s3_key = f"shards/{shard_name}"
    s3_client.upload_file(fname, BUCKET, s3_key)
    os.remove(fname)
    print(f"  Uploaded + removed {shard_name}")

def create_shards_s3(bucket_name, local_pattern='shards/geo-%06d.tar', max_count=1000):
      print("Fetching metadata from DynamoDB...")
      items = []
      response = table.scan()
      items.extend(response.get('Items', []))
      while 'LastEvaluatedKey' in response:
          response = table.scan(ExclusiveStartKey=response['LastEvaluatedKey'])
          items.extend(response.get('Items', []))
      print(f"Retrieved {len(items)} locations")

      os.makedirs('shards', exist_ok=True)

      writer = wds.ShardWriter(
          local_pattern,
          maxcount=max_count,
          post=upload_and_delete
      )

      skipped = 0
      for loc_data in tqdm(items):
          loc_id = loc_data['location_id']
          sample = {
              "__key__": loc_id,
              "json": json.dumps(loc_data).encode("utf-8")
          }

          images_found = 0
          for heading in [0, 90, 180, 270]:
              s3_key = f"raw-images/images/{loc_id}_h{heading}.jpg"
              try:
                  s3_response = s3_client.get_object(Bucket=bucket_name, Key=s3_key)
                  sample[f"h{heading}.jpg"] = s3_response['Body'].read()
                  images_found += 1
              except s3_client.exceptions.NoSuchKey:
                  continue
              except Exception as e:
                  print(f"Error fetching {s3_key}: {e}")

          if images_found == 4:
              writer.write(sample)
          else:
              skipped += 1

      writer.close()
      print(f"\nSharding complete. Skipped {skipped} incomplete locations.")

# Run sharding
#create_shards_s3(BUCKET, 'shards/geo-%06d.tar')


In [ ]:
response = s3_client.list_objects_v2(Bucket='geolocation-transformer-data', Prefix='shards/')
print(len([o for o in response.get('Contents', []) if o['Key'].endswith('.tar')]), "shards")

We have limited access to street view images because we are poor. So we need to stratify our sampling quite carefully.

Too much weight in big countries: n is too small in small countries, very little predictive power

Too much weight in small countries: big countries will be too diverse to learn effectively with such little imagery

To get an approximation of how "diverse" a country is, I think it is reasonable to use a measure of population * area (people use roads, and roads are used to connect area)

We then add a "temperature" constant in order to normalize, tuning this until... it looks somewhat reasonable. May be difficult to treat this as a real hyperparameter given we are poor.

In [ ]:
from google.colab import userdata
import pandas as pd
import numpy as np

# Add NICO_API_KEY and IAN_API_KEY via Colab Secrets (left sidebar → key icon).
# Never hardcode API keys here — this notebook may be shared or committed.
NICO_API_KEY = userdata.get('NICO_API_KEY')
IAN_API_KEY  = userdata.get('IAN_API_KEY')
API_KEY = NICO_API_KEY

# Change this path to where your 'Geolocation' folder is located on your local machine
DRIVE_PATH = '/content/drive/MyDrive/Geolocation'

CONFIG = {
    'total_images': 50000,
    'images_per_location': 4,
    'output_dir': os.path.join(DRIVE_PATH, 'images'),
    'metadata_path': os.path.join(DRIVE_PATH, 'metadata.csv'),
    'headings': [0, 90, 180, 270],  # N, E, S, W
    'image_size': '336x336',
    'fov': 90,
    'pitch': 0,
}

Path(CONFIG['output_dir']).mkdir(parents=True, exist_ok=True)
print(f"images will be saved to: {CONFIG['output_dir']}")

MAX_LOCATIONS = CONFIG['total_images'] // CONFIG['images_per_location']

supported_countries = pd.read_csv(os.path.join(DRIVE_PATH, 'Supported Countries.csv'), header=None, encoding='latin1').T


In [ ]:
supported_countries[0] = supported_countries[0].str.strip()

df = pd.read_csv(os.path.join(DRIVE_PATH, 'world_population.csv'))
df = df[['Country/Territory', '2022 Population', 'Area (km²)']]
df = df[df['Country/Territory'].isin(supported_countries[0])]

df['weight'] = (np.power(df['2022 Population'], 0.30) +
                np.power(df['Area (km²)'], 0.25) +
                350)

total_weight = df['weight'].sum()
df['normalized_weight'] = df['weight'] / total_weight

df['allocated_locations'] = (df['normalized_weight'] * MAX_LOCATIONS).astype(int)

COUNTRY_ALLOCATION_MAP = {}
for index, row in df.iterrows():
    COUNTRY_ALLOCATION_MAP[row['Country/Territory']] = row['allocated_locations']

# Create COUNTRIES_LIST from the keys of the map
COUNTRIES_LIST = list(COUNTRY_ALLOCATION_MAP.keys())

print(f"Total budget: {CONFIG['total_images']} images")
print(f"Max locations: {MAX_LOCATIONS}")
print(f"Countries being processed: {len(COUNTRIES_LIST)}")
print("\nAll allocations (sorted):")
for country in sorted(COUNTRY_ALLOCATION_MAP, key=COUNTRY_ALLOCATION_MAP.get, reverse=True):
    print(f"  {country}: {COUNTRY_ALLOCATION_MAP[country]} locations")


## Formula

The rationale above describes using `population × area` as a diversity proxy. The actual formula is:

$$\text{score}(c) = \text{pop}^{0.30} + \text{area}^{0.25} + \text{temperature}$$

**Why additive rather than multiplicative?**
A multiplicative formula (`pop × area`) causes large-population + large-area countries like India or Canada to consume nearly the entire budget. The additive structure keeps the two signals independent, and the flat `temperature` term added to every country guarantees a meaningful floor allocation even for micro-territories like Monaco or Jersey.

**What do the exponents control?**
- `pop^0.30` — strongly sub-linear. A 10× population increase only raises the score by ~2×, preventing billion-person countries from dominating.
- `area^0.25` — even more sub-linear. A 10× area increase raises the score by ~1.78×. Geographic spread matters, but less than population (roads need people to build and use them).

**What does temperature control?**
Raising `temperature` compresses the distribution toward uniform — every country gets closer to the same allocation. Lowering it sharpens the contrast, giving more weight to populous/large countries.

**Tuning philosophy:**
These values are heuristics, not derived from first principles. They are logged as W&B hyperparameters in the next cell so different strategies can be compared in a single dashboard.


In [ ]:
!pip install wandb -q

import wandb

wandb.init(
    project="streetview-stratification",
    config={
        "pop_exponent": 0.30,
        "area_exponent": 0.25,
        "temperature": 350,
        "total_images": CONFIG['total_images'],
        "max_locations": MAX_LOCATIONS,
        "num_countries": len(COUNTRIES_LIST),
    }
)

allocation_table = wandb.Table(
    dataframe=df[['Country/Territory', '2022 Population', 'Area (km²)', 'normalized_weight', 'allocated_locations']]
)
wandb.log({"country_allocation": allocation_table})
wandb.finish()
print("W&B run logged.")


In [ ]:
plot_df = df.sort_values('allocated_locations', ascending=False)

fig, ax = plt.subplots(figsize=(20, 6))
ax.bar(plot_df['Country/Territory'], plot_df['allocated_locations'], color='steelblue')
ax.set_xlabel('Country')
ax.set_ylabel('Allocated Locations')
ax.set_title(
    f'Street View Sampling Allocation by Country\n'
    f'pop^0.30 + area^0.25 + temp=350  |  '
    f'{len(plot_df)} countries  |  {plot_df["allocated_locations"].sum():,} total locations'
)
plt.xticks(rotation=90, fontsize=6)
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_PATH, 'stratification_allocation.png'), dpi=150)
plt.show()
print(f"Saved stratification_allocation.png to Drive")


In [ ]:
import datetime

STRATIFICATION_VERSION = "v1"

export_df = df[['Country/Territory', '2022 Population', 'Area (km²)', 'normalized_weight', 'allocated_locations']].copy()
export_df = export_df.sort_values('allocated_locations', ascending=False).reset_index(drop=True)

export_path = os.path.join(DRIVE_PATH, f'stratification_{STRATIFICATION_VERSION}.csv')
export_df.to_csv(export_path, index=False)

print(f"Saved {export_path}")
print(f"  Countries : {len(export_df)}")
print(f"  Locations : {export_df['allocated_locations'].sum():,} / {MAX_LOCATIONS:,}")
print(f"  Generated : {datetime.datetime.now().isoformat()}")
print(f"  Params    : pop_exp=0.30, area_exp=0.25, temperature=350")


# Naming
Next two sections get the list of countries from the geopandas Natural Earth dataset and map their naming conventions to ours.

For example we have "Eswatini" but they have "Swaziland".

---

## Dropped countries

The following countries are in our supported list but **do not exist in the Natural Earth dataset**, so we cannot generate geometry or sample points for them. They are excluded from the pipeline entirely.

| Country | Reason |
|---|---|
| Christmas Island | Not in Natural Earth dataset |
| US Virgin Islands | Not in Natural Earth dataset (use "U.S. Virgin Is." workaround exists but geometry is unreliable) |
| Curaçao | Not in Natural Earth dataset |

These account for a negligible fraction of the global street view coverage and are not worth the added complexity of a manual geometry workaround.


In [ ]:
# we are using geopandas for natural earth dataset
world = gpd.read_file(
    "https://d2ad6b4ur7yvpq.cloudfront.net/naturalearth-3.3.0/ne_50m_admin_0_countries.geojson"
)

print(f"Available columns: {world.columns.tolist()}")
print(f"Total countries in dataset: {len(world)}")

available = []
missing = []

for country in COUNTRIES_LIST:
    match = world[world['name'] == country]
    if len(match) > 0:
        available.append(country)
    else:
        missing.append(country)

print(f"\nFound: {len(available)} countries")
print(f"Missing: {len(missing)} countries")

if missing:
    print(f"\nMissing countries (need name mapping):")
    for m in missing:
        print(f"  - {m}")

In [ ]:
# Map our country names to Natural Earth names
NAME_MAPPING = {
    # Our name -> Natural Earth Dataset name
    "United Kingdom": "United Kingdom",
    "South Korea": "Korea",
    "North Macedonia": "Macedonia",
    "Czech Republic": "Czech Rep.",
    "Eswatini": "Swaziland",
    "Dominican Republic": "Dominican Rep.",
    "Laos": "Lao PDR",
    "Faroe Islands": "Faeroe Is.",
    "Northern Mariana Islands": "N. Mariana Is.",
    "US Virgin Islands": "U.S. Virgin Is.",
    "Curacao": "Curaçao"
}

print(world['name'].sort_values().tolist())

def get_country_geometry(country_name: str, world_gdf: gpd.GeoDataFrame):
    """Get geometry for a country, handling name variations"""
    # Try direct match first
    match = world_gdf[world_gdf['name'] == country_name]

    # Try mapping if no direct match
    if len(match) == 0:
        mapped_name = NAME_MAPPING.get(country_name)
        if mapped_name:
            match = world_gdf[world_gdf['name'] == mapped_name]

    # Try partial match as fallback
    if len(match) == 0:
        match = world_gdf[world_gdf['name'].str.contains(country_name, case=False, na=False)]

    if len(match) > 0:
        return match.geometry.values[0]

    return None

# Test
country_test = "Curaçao"
test_geom = get_country_geometry(country_test, world)
print(f"{country_test}: {test_geom.geom_type if test_geom else 'NOT FOUND'}")

# Point Generation
Next three cells generate random points contained inside a country's bounding box and return a valid Point(lat, lng) if streetview metadata is valid.

Note the radius of snapping to 10,000 meters so the actual point used is the snapped point

In [ ]:
# check with street view metadata with single point

def get_metadata_streetview(lat: float, lng: float, api_key: str):
  url = 'https://maps.googleapis.com/maps/api/streetview/metadata'
  params = {
      'location': f'{lat},{lng}',
      'key':  api_key,
      'radius': 5000,
      'source': 'outdoor'
  }

  try:
    response = requests.get(url, params=params)
    if response.status_code == 200:
      data = response.json()
      if data.get('status') == 'OK':
        return {
            'pano_id': data.get('pano_id'),
            'lat': data['location']['lat'],
            'lng': data['location']['lng'],
            'date': data.get('date')
        }
  except:
      pass

  return None

print(get_metadata_streetview('-21.4152485518145', '22.09998748488863', API_KEY))

In [ ]:
def random_point_in_polygon(geometry, max_attempts: int = 1000):
    """
    Generate a random point inside a polygon using rejection sampling.
    Works with both Polygon and MultiPolygon. This basically throws a
    dart inside given random country
    """

    # finds the furthest North, West, East, South points and draws a rectangle connecting them
    minx, miny, maxx, maxy = geometry.bounds

    for _ in range(max_attempts):
        # Random point in geometry bounding box
        lng = random.uniform(minx, maxx)
        lat = random.uniform(miny, maxy)

        point = Point(lng, lat)  # Shapely Point takes (x, y) = (lng, lat)

        if geometry.contains(point):
          metadata = get_metadata_streetview(lat, lng, API_KEY)
          if metadata:
            snapped_point = Point(metadata['lng'], metadata['lat'])
            if geometry.contains(snapped_point):
              return metadata
            else:
              print(f"Skipping: Point snapped across the border")

    return None

def generate_points_for_country( # this is how they write parameters in zon
    country_name: str,
    num_points: int,
    world_gdf: gpd.GeoDataFrame) -> list:
    """Generate n random points inside a country's borders"""
    geometry = get_country_geometry(country_name, world_gdf)

    if geometry is None:
        print(f"No geometry for {country_name}")
        return []

    points = []
    for i in range(num_points * 10):  # Try up to 10x to get enough points cuz most points will land in water
        point = random_point_in_polygon(geometry)
        if point:
            points.append(point)

        if len(points) >= num_points:
            break

    return points[:num_points]

# Test
test_points = generate_points_for_country("Botswana", 10, world)
print(f"Generated {len(test_points)} points for Botswana")
print(f"Sample: {test_points[:3]}")

In [ ]:
def visualize_country_points(country_name: str, points: list, world_gdf: gpd.GeoDataFrame):
    """Plot country with generated points"""
    geometry = get_country_geometry(country_name, world_gdf)

    if geometry is None:
        print(f"Cannot visualize {country_name}")
        return

    fig, ax = plt.subplots(figsize=(10, 8))

    # Plot country
    gdf = gpd.GeoDataFrame(geometry=[geometry])
    gdf.plot(ax=ax, color='lightblue', edgecolor='black')

    # Plot points
    lats = []
    lngs = []

    # points have structure: [{'pano_id': {pano_id}, 'lat': {lat}, 'lng': {lng}, 'date': {date}}, ...]
    for p in points:
      lat = p['lat']
      lng = p['lng']

      lats.append(lat)
      lngs.append(lng)

    ax.scatter(lngs, lats, c='red', s=20, zorder=5, label=f'{len(points)} points')

    ax.set_title(f'{country_name}')
    ax.legend()
    plt.show()

# Test visualization
test_points = generate_points_for_country("Switzerland", 10, world)
visualize_country_points("Switzerland", test_points, world)

# Generate valid points for all countries
Next 3 cells test generating points for all countries using a stratified distribution created in cell 2

In [ ]:
# generate points for all countries

def generate_all_country_points(countries_list, allocation_map, world_gdf):
  all_points = {}

  for country in tqdm(countries_list, desc="Generating points"):
    points_total = allocation_map.get(country, 0)

    if points_total <= 0:
      continue

    points = generate_points_for_country(country, points_total, world_gdf)

    for i, p in enumerate(points):
        p['location_id'] = f"{country}_{i}_{p['pano_id']}"

    all_points[country] = points

  return all_points

country_points = generate_all_country_points(COUNTRIES_LIST, COUNTRY_ALLOCATION_MAP, world)

print(f"\nTotal countries: {len(country_points)}")



In [ ]:
rows = []
for country, points in country_points.items():
    for p in points:
        row = p.copy()
        row['country'] = country
        rows.append(row)

df_points = pd.DataFrame(rows)

print("Total rows generated:", len(df_points))
print("\n--- Data Format Preview ---")
print(df_points.head())

save_path = os.path.join(DRIVE_PATH, 'generated_points_check.csv')
df_points.to_csv(save_path, index=False)
print(f"\n Saved to: {save_path}")

In [ ]:
test_country = 'Turkey'
print(country_points)
test_specific_country_points = country_points[test_country]
visualize_country_points(test_country, test_specific_country_points, world)

# Getting actual images

In [ ]:
from pathlib import Path
import os

def download_streetview_image(pano_id: str, heading: int, filepath: str) -> bool:
    """Download a single Street View image."""
    url = "https://maps.googleapis.com/maps/api/streetview"
    params = {
        'size': CONFIG['image_size'],
        'pano': pano_id, # we are using generated pano instead of (lat, lng) to get image
        'heading': heading,
        'pitch': CONFIG['pitch'],
        'fov': CONFIG['fov'],
        'key': API_KEY,
        'return_error_code': 'true',
        'radius': 5000
    }

    try:
        response = requests.get(url, params=params, timeout=10)

        if response.status_code == 200:
            content_type = response.headers.get('content-type', '')
            if 'image' in content_type:
                with open(filepath, 'wb') as f:
                    f.write(response.content)
                return True
            else:
                print(f"  Not an image: {content_type}")
                return False
        else:
            print(f"  HTTP error: {response.status_code}")
            return False

    except Exception as e:
        print(f"  Error: {e}")
        return False


def download_4_way_panorama(point: dict, output_dir: str) -> bool:
    success = True

    pano_id = point['pano_id']
    location_id = point['location_id']

    for heading in CONFIG['headings']:
        filename = f"{location_id}_h{heading}.jpg"
        filepath = os.path.join(output_dir, filename)

        if not download_streetview_image(pano_id, heading, filepath):
            success = False

        time.sleep(0.05)

    return success


def download_all_images(country_points: dict, output_dir: str) -> dict:
    successful_points = {}
    total_images = 0
    failed_locations = []

    total_locations = sum(len(pts) for pts in country_points.values())
    print(f"Starting download for {total_locations} locations")
    print(f"This will make {total_locations * 4} API calls")
    print(f"Estimated cost: ${total_locations * 4 * 0.007:.2f}")
    print(f"{'='*50}\n")

    for country, points in tqdm(country_points.items(), desc="Countries"):
        successful_points[country] = []

        for point in tqdm(tqdm(points, desc=f"  {country}", leave=False)):
            if download_4_way_panorama(point, output_dir):
                successful_points[country].append(point)
                total_images += 4
            else:
                failed_locations.append(point['location_id'])

        print(f"  {country}: {len(successful_points[country])}/{len(points)} locations")

    print(f"\n{'='*50}")
    print(f"Download complete!")
    print(f"Total images downloaded: {total_images}")
    print(f"Failed locations: {len(failed_locations)}")

    if failed_locations:
        print(f"Failed: {failed_locations[:10]}{'...' if len(failed_locations) > 10 else ''}")

    return successful_points

In [ ]:
def save_metadata(country_points: dict, filepath: str):
    """
    Save metadata CSV for training.

    Creates CSV with columns: location_id, country, lat, lng
    """
    rows = []

    for country, points in country_points.items():
        for p in points:
            rows.append({
                'location_id': p['location_id'],
                'country': country,
                'lat': p['lat'],
                'lng': p['lng'],
                'date': p.get('date', 'unknown'),
                'pano_id': p.get('pano_id', 'unknown')
            })

    df_new = pd.DataFrame(rows)

    if os.path.exists(filepath):
        df_existing = pd.read_csv(filepath)
        df = pd.concat([df_existing, df_new], ignore_index=True)
        df = df.drop_duplicates(subset=['location_id'])  # avoid duplicates
        print(f"Appended {len(df_new)} new locations to existing {len(df_existing)}")
    else:
        df = df_new

    df.to_csv(filepath, index=False)
    print(f"Saved metadata for {len(df)} total locations to {filepath}")
    return df

In [ ]:
CALL_PHOTOS_API = True  # Set to True ONLY when ready to spend money

if CALL_PHOTOS_API:
    total_locations = sum(len(pts) for pts in country_points.values())
    total_cost = total_locations * 4 * 0.007

    print(f"WARNING: This will make {total_locations * 4} PAID API calls")
    print(f"Estimated cost: ${total_cost:.2f}")
    print()

    confirm = input("Type 'yes' to proceed: ")

    if confirm.lower() == 'yes':
        successful_points = download_all_images(
            country_points,
            CONFIG['output_dir']
        )

        df_metadata = save_metadata(
            successful_points,
            CONFIG['metadata_path']
        )

        print(f"\n Done: Images saved to: {CONFIG['output_dir']}")
        print(f" Metadata saved to: {CONFIG['metadata_path']}")
    else:
        print(" Cancelled")

else:
    print("CALL_PHOTOS_API is False - no API calls made")
    print("Set CALL_PHOTOS_API = True when ready to download images")

    # Show what would be downloaded
    total_locations = sum(len(pts) for pts in country_points.values())
    print(f"\nReady to download:")
    print(f"  Locations: {total_locations}")
    print(f"  Images: {total_locations * 4}")
    print(f"  Estimated cost: ${total_locations * 4 * 0.007:.2f}")

In [ ]:
def verify_downloads(metadata_path: str, image_dir: str):
    df = pd.read_csv(metadata_path)

    missing = []
    for _, row in df.iterrows():
        for heading in [0, 90, 180, 270]:
            filepath = os.path.join(image_dir, f"{row['location_id']}_h{heading}.jpg")
            if not os.path.exists(filepath):
                missing.append(filepath)

    if missing:
        print(f"Missing {len(missing)} images:")
    else:
        print(f"All {len(df) * 4} images present")

verify_downloads(CONFIG['metadata_path'], CONFIG['output_dir'])